In [ ]:
import pandas as pd
from rapidfuzz.fuzz import ratio
import re
import src_any_abogados_fuzzy_merge_df_er_v1_sec as src
import src_any_abogados_numbers_merge_df_er_v2_sec as src2

In [2]:
def remove_empty_columns(df):

    df_aux = df.copy()
    empty_columns = list(df_aux.columns[df_aux.isna().mean() == 1])   # Almacena en una lista las columnas que están totalmente vacias
    if "Unnamed: 0" in df_aux.columns:
        empty_columns.append("Unnamed: 0")
    df_aux = df_aux.drop(columns=empty_columns)
    return df_aux

In [5]:
#Lectura de tablas
df_gm_o = pd.read_csv("datasets/Dataset_GM_PR.csv")
df_ca_o = pd.read_csv("datasets/datos_abogados.csv") 
df_naics_o = pd.read_csv("datasets/NAICS_Puerto Rico.csv")

df_gm_o = remove_empty_columns(df_gm_o)
df_ca_o = remove_empty_columns(df_ca_o)
df_naics_o = remove_empty_columns(df_naics_o)

In [183]:
#Agregar columna "is Firm" para diferenciar Firma de Abogados
df_gm_o['is Firm'] = df_gm_o['Name'].apply(lambda x: bool(re.search(src.pattern_firma, str(x), flags=re.IGNORECASE)))

In [ ]:
#DataFrame que almacena los registros que son firmas
df_gm_firmas = df_gm_o[df_gm_o["is Firm"]]

#DataFrame filtrado y almacena los registro que son abogados
df_gm_o = df_gm_o[~df_gm_o["is Firm"]]

#Normalización de nombres
df_gm_o["Clean_Name"] = df_gm_o["Name"].apply(lambda x: src.name_normalized(x)).str.lower()
df_ca_o["FULL NAME"] = df_ca_o["FULL NAME"].apply(lambda x: src.name_normalized(x)).str.lower()
df_naics_o["Name_N"] = df_naics_o["Name_N"].apply(lambda x: src.name_normalized(x)).str.lower()

#Reset index
df_gm_o = df_gm_o.reset_index(drop=True)
df_ca_o = df_ca_o.reset_index(drop=True)
df_naics_o = df_naics_o.reset_index(drop=True)

In [8]:
# Columnas importantes: Name, city, state_name, Address, Website,Phone,Google_category,Type,Email,Specialization,Education,Experience,
# Columnas a Analizar: zip, state_id, Orig Specialization
# NO Columnas importantes: Google_URL, population,Google_rank,Google_opinions,Google_category,Processed,Name AI

df_gm_o.columns 


Index(['Name', 'Google_URL', 'zip', 'city', 'state_id', 'state_name',
       'population', 'Address', 'Website', 'Phone', 'Google_rank',
       'Google_opinions', 'Google_category', 'Name len', 'Type', 'Email',
       'Orig Specialization', 'Specialization', 'Education', 'Experience',
       'Processed', 'Name AI'],
      dtype='object')

In [ ]:
#Separar columna de nombres para realizar las comparciones difusas
df_gm = df_gm_o[["Clean_Name"]]
df_ca = df_ca_o[["FULL NAME"]]
df_naics = df_naics_o[["Name_N"]]

In [185]:
ff = pd.concat([df_gm, df_ca, df_naics], axis=1)
ff.shape[1]

list(ff.columns)

['Clean_Name', 'FULL NAME', 'Name_N']

In [186]:
#res = src.perfect_matches(pd.concat([df_gm, df_ca, df_naics], axis=1))
#res

In [187]:
df_gm_o.isna().mean()*100

Name                    0.000000
Google_URL              0.000000
zip                     0.000000
city                    0.000000
state_id                0.000000
state_name              0.000000
population              0.000000
Address                 6.733524
Website                65.902579
Phone                   8.595989
Google_rank            19.340974
Google_opinions        17.765043
Google_category         2.148997
Name len                0.000000
Type                    0.000000
Email                  22.636103
Orig Specialization    22.636103
Specialization         22.636103
Education              22.636103
Experience             22.636103
Processed              22.636103
Name AI                22.636103
is Firm                 0.000000
Clean_Name              0.000000
dtype: float64

In [188]:
df_ca_o[["tel_residencial","tel_oficina","tel_celular"]].isna().mean()*100

tel_residencial    99.407407
tel_oficina        91.407407
tel_celular        95.703704
dtype: float64

In [189]:
# Mostrar todas las filas
pd.set_option("display.max_rows", None)

columnas_con_mas_99_nan = df_naics_o.isna().mean() * 100
print(columnas_con_mas_99_nan)

ZoomInfo Contact ID                           0.000000
Last Name                                     0.000000
First Name                                    0.000000
Middle Name                                  51.747881
Job Title                                    12.764831
Job Title Hierarchy Level                    31.991525
Management Level                             68.697034
Job Start Date                                1.800847
Job Function                                 46.239407
Department                                   39.671610
Company Division Name                       100.000000
Direct Phone Number                          64.565678
Email Address                                25.317797
Email Domain                                 25.370763
Mobile phone                                 61.281780
Highest Level of Education                   65.836864
Contact Accuracy Score                        0.000000
Contact Accuracy Grade                        0.000000
ZoomInfo C

In [190]:
df_naics_o.columns

Index(['ZoomInfo Contact ID', 'Last Name', 'First Name', 'Middle Name',
       'Job Title', 'Job Title Hierarchy Level', 'Management Level',
       'Job Start Date', 'Job Function', 'Department', 'Company Division Name',
       'Direct Phone Number', 'Email Address', 'Email Domain', 'Mobile phone',
       'Highest Level of Education', 'Contact Accuracy Score',
       'Contact Accuracy Grade', 'ZoomInfo Contact Profile URL',
       'LinkedIn Contact Profile URL', 'Person Street', 'Person City',
       'Person State', 'Person Zip Code', 'Country', 'ZoomInfo Company ID',
       'Company Name', 'Website', 'Founded Year', 'Company HQ Phone', 'Fax',
       'Ticker', 'Revenue (in 000s USD)', 'Revenue Range (in USD)',
       'Employees', 'Employee Range', 'SIC Code 1', 'SIC Code 2', 'SIC Codes',
       'NAICS Code 1', 'NAICS Code 2', 'NAICS Codes', 'Primary Industry',
       'Primary Sub-Industry', 'All Industries', 'All Sub-Industries',
       'Industry Hierarchical Category',
       'Seconda

In [191]:
#df_ca_o["telefono"] = df_ca_o["tel_celular"].combine_first(df_ca_o["tel_residencial"]).combine_first(df_ca_o["tel_oficina"])
df_ca_o = src2.combinar_columnas_prioridad(df_ca_o,["tel_celular","tel_residencial","tel_oficina"],"telefono")

In [192]:
df_ca_o["telefono"] = df_ca_o["telefono"].apply(lambda phone: src2.normalized_phone(str(phone)) if pd.notnull(phone) else None)
df_naics_o["Mobile phone"] = df_naics_o["Mobile phone"].apply(lambda phone: src2.normalized_phone(str(phone)) if pd.notnull(phone) else None)

In [193]:
# Para la columna 'telefono'
telefonos_duplicados = df_ca_o["telefono"].value_counts()
telefonos_duplicados = telefonos_duplicados[telefonos_duplicados > 1]

print(telefonos_duplicados)

# Para la columna 'Mobile phone'
mobiles_duplicados = df_naics_o["Mobile phone"].value_counts()
mobiles_duplicados = mobiles_duplicados[mobiles_duplicados > 1]

print(mobiles_duplicados)

telefono
7872810707    4
7872899250    2
7877511912    2
7877599292    2
7877519040    2
7877634111    2
Name: count, dtype: int64
Mobile phone
7874572475    2
7874032566    2
Name: count, dtype: int64


In [194]:
df_ca_o[df_ca_o["telefono"] == "7872810707"]

,FULL NAME,FNAME,LNAME,colegiacion,rua,correo,tel_residencial,tel_oficina,tel_celular,otro,especialidades,Practice Area,delegacion,State,telefono
230,camila rivera negron,Camila,Rivera Negron,20862.0,22055.0,camila@therivera.group,NaN,787-281-0707,NaN,NaN,"Administrativo,Civil,Litigación,Procedimiento ...",;Administrativo;Civil;Litigación;Procedimiento...,Delegación de Bayamón,Bayamón,7872810707
324,lizabel negron vargas,Lizabel,Negron Vargas,10961.0,9758.0,NaN,NaN,787-281-0707,NaN,NaN,"Apelativa,Civil,Contratos,Corporativo,Federal,...",;Apelativa;Civil;Contratos;Corporativo;Federal...,Delegación de Bayamón,Bayamón,7872810707
336,edgardo rivera rivera,Edgardo,Rivera Rivera,11112.0,9884.0,edgardo@THERIVERA.group,NaN,787-281-0707,NaN,NaN,"Apelativa,Civil,Contratos,Daños y perjuicios,F...",;Apelativa;Civil;Contratos;Daños y perjuicios;...,Delegación de San Juan,San Juan,7872810707
433,gabriela rivera negron,Gabriela,Rivera Negron,20746.0,21820.0,NaN,NaN,787-281-0707,NaN,NaN,"Civil,Contratos,Corporativo,Daños y perjuicios...",;Civil;Contratos;Corporativo;Daños y perjuicio...,Delegación de Bayamón,Bayamón,7872810707


In [195]:
df_ca_o_unique = df_ca_o.drop_duplicates(subset="telefono")
df_naics_o_unique = df_naics_o.drop_duplicates(subset="Mobile phone")

df_merged = src2.merge_by_contact_number(df_ca_o_unique, df_naics_o_unique, "telefono", "Mobile phone", "FULL NAME", "Name_N")
df_merged.head()


,FULL NAME,FNAME,LNAME,colegiacion,rua,correo,tel_residencial,tel_oficina,tel_celular,otro,...,Body New,Context,Subject 2 New,Body 2 New,Subject 3 New,Body 3 New,Subject 4 New,Body 4 New,Name_N,score
0,maria barrera rosario,Maria,Barrera Rosario,9731.0,8494.0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,larry suckle,0.303030
1,esteban mujica cotto,Esteban,Mujica Cotto,10082.0,8808.0,esteban.mujica@mcalawfirm.com,NaN,NaN,787-518-4101,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,esteban mujicacotto,0.974359
2,josue castellanos otero,Josue,Castellanos Otero,20500.0,18329.0,castellanos@lawyerspsc.com,NaN,787-529-6787,787-299-5935,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,josue emanuel castellanos otero,0.851852
3,francisco garcia garcia,Francisco,Garcia Garcia,9757.0,8510.0,fgarcia@amgprlaw.com,787-398-4898,787-281-1800,787-398-4898,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,francisco garcia,0.820513


In [196]:
print(len(df_ca_o))
print(len(df_naics_o))
print(len(df_merged))

2700
1888
4


In [197]:
# 1. Obtener el nombre de la última columna
penultima_columna = df_merged.columns[-2]
ultima_columna = df_merged.columns[-1]


# 2. Reordenar las columnas según tu orden deseado
columnas_nuevas = (
    [df_merged.columns[0]] +               # primera columna original
    [penultima_columna] +                  # la última como segunda
    [ultima_columna] +                     # la penúltima como tercera
    ["telefono", "Mobile phone"] +         # luego las dos que especificaste
    [col for col in df_merged.columns      # el resto de columnas
     if col not in [df_merged.columns[0], penultima_columna, "telefono", "Mobile phone",ultima_columna]]
)

# 3. Reordenar el DataFrame
df_merged = df_merged[columnas_nuevas]
df_merged.shape

(4, 111)

In [198]:
telefonos_matcheados = df_merged["telefono"].unique()
df_ca_sin_match = df_ca_o[~df_ca_o["telefono"].isin(telefonos_matcheados)].copy()
df_naics_sin_match = df_naics_o[~df_naics_o["Mobile phone"].isin(telefonos_matcheados)].copy()

# Asegúrate de que las columnas existan para poder concatenar
df_ca_sin_match["Mobile phone"] = None
df_ca_sin_match["Name_N"] = None
df_ca_sin_match["score"] = 0

# 1. Obtener las columnas del df_merged (resultado final deseado)
columnas = df_merged.columns.tolist()

# 2. Asegurar que df_ca_sin_match tenga todas esas columnas
for col in columnas:
    if col not in df_ca_sin_match.columns:
        df_ca_sin_match[col] = None  # o np.nan si prefieres

# 3. Reordenar columnas (ya sin miedo al KeyError)
df_ca_sin_match = df_ca_sin_match[columnas]

# 4. Concatenar
df_final = pd.concat([df_merged, df_ca_sin_match], ignore_index=True)

# Ordenar columnas como en df_merged
columnas = df_merged.columns.tolist()
df_ca_sin_match = df_ca_sin_match[columnas]
df_final = pd.concat([df_merged, df_ca_sin_match], ignore_index=True)
df_final.shape




/var/folders/t0/47zrf3xx0ll0tvqnt63jt_980000gn/T/ipykernel_45035/1651953900.py:22: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_final = pd.concat([df_merged, df_ca_sin_match], ignore_index=True)
/var/folders/t0/47zrf3xx0ll0tvqnt63jt_980000gn/T/ipykernel_45035/1651953900.py:27: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_final = pd.concat([df_merged, df_ca_sin_match], ignore_index=True)


(307, 111)

In [199]:
df_final.to_csv("datasets/merged_by_phone_number.csv")


In [200]:
df_final.head()

,FULL NAME,Name_N,score,telefono,Mobile phone,FNAME,LNAME,colegiacion,rua,correo,...,Ready,Subject New,Body New,Context,Subject 2 New,Body 2 New,Subject 3 New,Body 3 New,Subject 4 New,Body 4 New
0,maria barrera rosario,larry suckle,0.303030,None,None,Maria,Barrera Rosario,9731.0,8494.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,esteban mujica cotto,esteban mujicacotto,0.974359,7875184101,7875184101,Esteban,Mujica Cotto,10082.0,8808.0,esteban.mujica@mcalawfirm.com,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,josue castellanos otero,josue emanuel castellanos otero,0.851852,7872995935,7872995935,Josue,Castellanos Otero,20500.0,18329.0,castellanos@lawyerspsc.com,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,francisco garcia garcia,francisco garcia,0.820513,7873984898,7873984898,Francisco,Garcia Garcia,9757.0,8510.0,fgarcia@amgprlaw.com,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,orlando cabrera rodriguez,None,0.000000,7874097891,None,Orlando,Cabrera Rodriguez,11655.0,10316.0,despacholegalox@gmail.com,...,None,None,None,None,None,None,None,None,None,None
